<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Colab_YOLO_Supervision_ByteTrack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO + Supervision + ByteTrack

In [ ]:

!pip install -U supervision ultralytics trackers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.7/224.7 kB 1.3 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded=files.upload()
VIDEO_PATH=list(uploaded.keys())[0]

Saving Gente caminando.mp4 to Gente caminando.mp4


In [ ]:

import os
import supervision as sv
from ultralytics import YOLO
from trackers import ByteTrackTracker
from tqdm.auto import tqdm
from google.colab import files


# CONFIGURACIÓN

VIDEO_PATH = "/content/Gente caminando.mp4"
OUTPUT_PATH = "/content/resultado_tracking.mp4"

if not os.path.isfile(VIDEO_PATH):
    raise FileNotFoundError(
        f"No se encontró el video: {VIDEO_PATH}"
    )

# Modelo YOLO
model = YOLO("yolo26n.pt")

# Información del video
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)

# Tracker actualizado
tracker = ByteTrackTracker()

# Anotadores
box = sv.BoxAnnotator()
label = sv.LabelAnnotator()
trace = sv.TraceAnnotator(trace_length=50)


# PROCESAMIENTO

with sv.VideoSink(OUTPUT_PATH, video_info) as sink:

    frames = sv.get_video_frames_generator(VIDEO_PATH)

    for frame in tqdm(
        frames,
        total=video_info.total_frames,
        desc="Procesando video"
    ):
        # Detectar objetos
        result = model(frame, verbose=False)[0]

        detections = sv.Detections.from_ultralytics(result)

        # Asignar IDs persistentes
        detections = tracker.update(detections)

        # Crear etiquetas
        labels = []

        if detections.tracker_id is not None:
            for tid, cid, conf in zip(
                detections.tracker_id,
                detections.class_id,
                detections.confidence
            ):
                class_name = model.names[int(cid)]
                confidence = float(conf)

                labels.append(
                    f"ID:{int(tid)} {class_name} {confidence:.2f}"
                )

        # Dibujar trayectorias y cajas
        annotated = frame.copy()
        annotated = trace.annotate(annotated, detections)
        annotated = box.annotate(annotated, detections)
        annotated = label.annotate(
            annotated,
            detections,
            labels=labels
        )

        # Guardar frame
        sink.write_frame(annotated)

print(f"Video terminado: {OUTPUT_PATH}")

# Descargar resultado
files.download(OUTPUT_PATH)


Procesando video:   0%|          | 0/598 [00:00<?, ?it/s]

Video terminado: /content/resultado_tracking.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('resultado_tracking.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
!pip install -U supervision ultralytics trackers

import os
import numpy as np
import supervision as sv
from ultralytics import YOLO
from trackers import ByteTrackTracker
from tqdm.auto import tqdm
from google.colab import files

# CONFIGURACIÓN
VIDEO_PATH = "/content/Gente caminando.mp4"
OUTPUT_PATH = "/content/resultado_tracking_id1.mp4"
TARGET_ID = 3

if not os.path.isfile(VIDEO_PATH):
    raise FileNotFoundError(f"No se encontró el video: {VIDEO_PATH}")

# Modelo YOLO y Tracker
model = YOLO("yolo26n.pt")
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
tracker = ByteTrackTracker()

# Anotadores
box = sv.BoxAnnotator()
label = sv.LabelAnnotator()
trace = sv.TraceAnnotator(trace_length=50)

with sv.VideoSink(OUTPUT_PATH, video_info) as sink:
    frames = sv.get_video_frames_generator(VIDEO_PATH)

    for frame in tqdm(frames, total=video_info.total_frames, desc="Procesando video (Solo ID 1)"):
        result = model(frame, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(result)

        # Actualizar tracker
        detections = tracker.update(detections)

        # Filtrar para mantener únicamente el ID seleccionado
        if detections.tracker_id is not None:
            mask = detections.tracker_id == TARGET_ID
            detections = detections[mask]

        # Crear etiquetas personalizadas si el ID objetivo está presente
        labels = []
        if detections.tracker_id is not None and len(detections.tracker_id) > 0:
            for tid, cid, conf in zip(detections.tracker_id, detections.class_id, detections.confidence):
                class_name = model.names[int(cid)]
                labels.append(f"ID:{int(tid)} {class_name} {float(conf):.2f}")

        # Anotar solo las detecciones filtradas
        annotated = frame.copy()
        if len(detections) > 0:
            annotated = trace.annotate(annotated, detections)
            annotated = box.annotate(annotated, detections)
            annotated = label.annotate(annotated, detections, labels=labels)

        sink.write_frame(annotated)

print(f"Video terminado con filtro de ID: {OUTPUT_PATH}")
files.download(OUTPUT_PATH)

Procesando video (Solo ID 1):   0%|          | 0/598 [00:00<?, ?it/s]

Video terminado con filtro de ID: /content/resultado_tracking_id1.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("bytetrack_explicacion.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
import numpy as np

W,H = 1280,720

video = cv2.VideoWriter(
    "supervision_explicacion.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    30,
    (W,H)
)

font=cv2.FONT_HERSHEY_SIMPLEX

for frame in range(300):

    img=np.ones((H,W,3),dtype=np.uint8)*255

    cv2.putText(
        img,
        "Flujo de trabajo de Supervision",
        (250,60),
        font,
        1.2,
        (0,0,0),
        3
    )

    # VIDEO

    if frame>10:

        cv2.rectangle(img,(80,180),(300,500),(0,0,0),3)

        cv2.putText(
            img,
            "VIDEO",
            (145,350),
            font,
            1,
            (0,0,0),
            3
        )

    # Flecha

    if frame>40:

        cv2.arrowedLine(
            img,
            (300,340),
            (430,340),
            (255,0,0),
            3
        )

    # YOLO

    if frame>70:

        cv2.rectangle(img,(430,180),(650,500),(0,180,0),3)

        cv2.putText(img,
                    "YOLO",
                    (500,240),
                    font,
                    1,
                    (0,120,0),
                    3)

        cv2.putText(img,
                    "Persona",
                    (470,300),
                    font,
                    .7,
                    (0,0,0),
                    2)

        cv2.putText(img,
                    "Auto",
                    (470,340),
                    font,
                    .7,
                    (0,0,0),
                    2)

        cv2.putText(img,
                    "Perro",
                    (470,380),
                    font,
                    .7,
                    (0,0,0),
                    2)

    # Flecha

    if frame>110:

        cv2.arrowedLine(
            img,
            (650,340),
            (810,340),
            (255,0,0),
            3
        )

    # SUPERVISION

    if frame>140:

        cv2.rectangle(
            img,
            (810,180),
            (1180,560),
            (255,120,0),
            3
        )

        cv2.putText(
            img,
            "SUPERVISION",
            (865,220),
            font,
            .9,
            (0,0,200),
            3
        )

        funciones=[
            "Detections",
            "ByteTrack",
            "BoxAnnotator",
            "LabelAnnotator",
            "TraceAnnotator",
            "LineZone",
            "PolygonZone",
            "VideoSink"
        ]

        for i,f in enumerate(funciones):

            color=(0,0,0)

            if frame>150+i*15:
                color=(0,150,0)

            cv2.putText(
                img,
                f,
                (850,270+i*35),
                font,
                .65,
                color,
                2
            )

    # Mensaje final

    if frame>260:

        cv2.putText(
            img,
            "Supervision organiza y visualiza las detecciones",
            (180,650),
            font,
            .9,
            (0,0,255),
            3
        )

    video.write(img)

video.release()

print("Video generado.")

Video generado.
